# Analyse et préparation des données des arbres

In [ ]:
import numpy as np
import pandas as pd
import itertools

## Chargement des données d'inventaire
* État de l'inventaire des arbres

In [ ]:
etat_inventaire_arbres = pd.read_csv(
    'etatinventaire_arbrespublics_pararrondissement.csv',
    index_col='Numéro '
)

print(etat_inventaire_arbres.shape)
print(etat_inventaire_arbres.columns)

* Toutes les données

In [ ]:
arbres_df = pd.read_csv(
    'arbres-publics.csv',
    dtype={col_name: 'str' for col_name in [
        'CODE_PARC',
        'Distance_ligne_rue',
        'LOCALISATION',
        'Localisation_code',
        'NOM_PARC',
        'Nom_secteur',
        'No_civique',
        'Rue',
        'Rue_a',
        'Rue_cote',
        'Rue_de',
        'Stationnement_jour',
        'Stationnement_heure',
    ]}
)

print(arbres_df.shape)
print(arbres_df.groupby('INV_TYPE').size().to_frame().T)
arbres_df.notna().sum().sort_values()

### Analyse et nettoyage des arrondissements
* Comparaison entre les informations des deux DataFrames
* À numéro d'arrondissement égal, on a le même nom, ou presque

In [ ]:
arrond_inv = etat_inventaire_arbres['Arrondissement'].to_frame()

arrond_df = arbres_df[['ARROND', 'ARROND_NOM']].drop_duplicates()
arrond_df = arrond_df.set_index('ARROND')

arrond_comp = pd.concat([arrond_inv, arrond_df], axis='columns')
arrond_comp['Pareil'] = \
    arrond_comp['ARROND_NOM'] == arrond_comp['Arrondissement']
arrond_comp

* Nettoyer l'information de l'arrondissement

In [ ]:
# Enlever l'information redondante
if 'ARROND_NOM' in arbres_df.columns:
    arbres_df.drop(columns='ARROND_NOM', inplace=True)

# Renommer la colonne
if 'ARROND' in arbres_df.columns:
    arbres_df.rename(columns={'ARROND': 'arrond_id'}, inplace=True)

arbres_df.columns

* Nettoyager et sauvegarder l'état de l'inventaire

In [ ]:
etat_inventaire_arbres.index.name = \
    etat_inventaire_arbres.index.name.strip()

etat_inventaire_arbres.to_csv('arbres_inventaire.csv')

## Analyse des autres colonnes redondantes
* Aperçu des colonnes les plus correlées

In [ ]:
factors = \
    arbres_df.transform(lambda c: c.factorize()[0])

{
    (np.abs(factors[c1] - factors[c2]).sum(), c1, c2)
    for c1, c2 in itertools.combinations(factors.columns, 2)
    if np.abs(factors[c1] - factors[c2]).sum() < 10 * len(factors)
}

### Analyse et nettoyage des essences d'arbre

In [ ]:
essences_df = arbres_df[
    ['Sigle', 'Essence_latin', 'Essence_ang', 'Essence_fr']
].drop_duplicates().sort_values('Sigle')

essences_df

* Vérifier le nombre de valeurs différentes par colonne

In [ ]:
{c: essences_df[c].nunique() for c in essences_df.columns}

* Valeurs redondantes dans `Essence_ang`

In [ ]:
essences_df[
    essences_df['Essence_ang'].duplicated(keep=False)
].sort_values('Essence_ang')

* Valeurs redondantes dans `Essence_fr`

In [ ]:
essences_df[
    essences_df['Essence_fr'].duplicated(keep=False)
].sort_values('Essence_fr')

* Sauvegarder l'information redondante ailleurs

In [ ]:
essences_df.to_csv('arbres_essences.csv', index=False)

* Enlever l'information redondante sur les essences

In [ ]:
for c in essences_df.columns[1:]:
    if c in arbres_df.columns:
        arbres_df.drop(columns=c, inplace=True)

arbres_df.columns

### Analyse des dépendances entre les colonnes
* Décompte des différentes valeurs par colonne

In [ ]:
nuniques = {
    c: arbres_df[c].nunique(dropna=False)
    for c in arbres_df.columns
}
nuniques

* Pour chaque paire de colonnes, calculer les différentes paires de
  valeurs et afficher les colonnes où le nombre de paires est était
  égal au nombre de valeurs différentes d'une seule colonne.
  * Ainsi, la colonne avec le plus de valeurs différentes peut
    déterminer la valeur unique de l'autre colonne.

In [ ]:
factors = \
    arbres_df.transform(lambda c: c.factorize()[0])

{
    (factors[[c1, c2]].drop_duplicates().shape[0], c1, c2)
    for c1, c2 in itertools.combinations(factors.columns, 2)
    if max(nuniques[c1], nuniques[c2]) ==
        factors[[c1, c2]].drop_duplicates().shape[0]
}

### Analyse et nettoyage des parcs
Les parcs sont définis seulement lorsque
`arbres_df['INV_TYPE'] == 'H'`.
* Les champs `'Code_secteur'` et `'Nom_secteur'` sont aussi définis
  uniquement avec les parcs

In [ ]:
parcs_df = arbres_df[
    ['CODE_PARC', 'NOM_PARC']
].drop_duplicates().sort_values('CODE_PARC')

parcs_df[
    parcs_df['NOM_PARC'].duplicated(keep=False)
].sort_values('NOM_PARC')

* Sauvegarder l'information sur les parcs

In [ ]:
parcs_df.dropna().to_csv('arbres_parcs.csv', index=False)

* Enlever l'information redondante sur les parcs

In [ ]:
for c in ['NOM_PARC', 'Code_secteur', 'Nom_secteur']:
    if c in arbres_df.columns:
        arbres_df.drop(columns=c, inplace=True)

arbres_df.columns

### Analyse et nettoyage des rues
Les rues sont définies seulement lorsque `arbres_df['INV_TYPE'] == 'R'`.

In [ ]:
sorted(arbres_df['Rue'].dropna().unique())[-12:]

In [ ]:
sorted(arbres_df['Rue_de'].dropna().unique())[-12:]

In [ ]:
sorted(arbres_df['Rue_a'].dropna().unique())[-12:]

* Les noms de rue ayant un différent format entre ces trois
  colonnes, on peut couper dans les données des arbres de rue.
  Cela inclut plusieurs colonnes qui ont des valeurs non définies
  lorsque `INV_TYPE` n'est pas `'R'`.

In [ ]:
arbres_df = arbres_df[arbres_df['INV_TYPE'] != 'R']

colonnes_rue = [
    'No_civique', 'Rue', 'Rue_cote', 'Rue_de', 'Rue_a',
    'Distance_pave', 'Distance_ligne_rue',
    'District', 'LOCALISATION', 'Localisation_code',
    'Stationnement_jour', 'Stationnement_heure',
]

for c in colonnes_rue:
    if c in arbres_df.columns:
        arbres_df.drop(columns=c, inplace=True)

arbres_df.columns

### Reconstituer l'historique des arbres
* Chercher une redondance des coordonnées GPS pour trouver les arbres
  rapportés plusieurs fois.

In [ ]:
arbres_df[['Longitude', 'Latitude']].drop_duplicates()

* Puisque presque toutes les coordonnées sont différentes,
  chercher des propriétés qui se répètent.

In [ ]:
test_df = arbres_df[
    ['arrond_id', 'CODE_PARC', 'Sigle', 'Longitude', 'Latitude']
].sort_values(
    ['arrond_id', 'CODE_PARC', 'Sigle']
)

arbres_df.loc[
    test_df[
        test_df.duplicated(keep=False)
    ].index,
    [
        'arrond_id', 'CODE_PARC', 'Sigle',
        'Longitude', 'Latitude', 'Date_Releve', 'DHP'
    ]
]

* Étant donné que les dates se répètent, nous n'avons pas d'historique
  par arbre. Chaque enregistrement est donc un arbre différent.